# Ad Click Volume Forecasting

## Project Overview

This project forecasts **daily ad click volume** over a 14-day horizon.
We generate synthetic daily click data for an advertising platform spanning 2 years
with weekly seasonality, click-through rate variations, and seasonal patterns.

**Models**: Naive baselines, LazyPredict (tabular), FLAML AutoML, StatsForecast.
**Forecast horizon**: 14 days ahead.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily ad click counts, predict the next 14 days. Click volume forecasts drive revenue projections (CPC billing), fraud detection baselines, and advertiser performance expectations.

## Why This Project Matters

Ad click forecasting is essential for CPC-based advertising revenue. It sets baseline expectations for fraud detection (anomalous click patterns), advertiser billing, and platform revenue guidance.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily ad clicks
- Weekly seasonality (higher weekdays)
- Growth trend (more users, more clicks)
- Seasonal CTR variation (higher in Q4 due to holiday shopping)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `clicks` | Daily ad click count (thousands) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'clicks'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}, freq={FREQ}")

Config: 730 points, horizon=14, season=7, freq=D


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

trend = 500 + 0.5 * t  # thousands
weekly = 80 * np.sin(2 * np.pi * (t + 1) / 7)  # weekday higher

# Q4 seasonal boost (higher CTR during shopping season)
seasonal = np.zeros(N_POINTS)
for i in range(N_POINTS):
    m = dates[i].month
    if m >= 10:
        seasonal[i] = 100 + (m - 10) * 40  # ramp up Oct-Dec

noise = np.random.normal(0, 50, N_POINTS)
clicks = trend + weekly + seasonal + noise
clicks = np.maximum(clicks, 50).round(0).astype(int)

df = pd.DataFrame({'date': dates, 'clicks': clicks})
print(f"Dataset shape: {df.shape}")
print(f"Unit: thousands of clicks")
df.head()

Dataset shape: (730, 2)
Unit: thousands of clicks


,date,clicks
0,2022-01-01,587
1,2022-01-02,572
2,2022-01-03,568
3,2022-01-04,543
4,2022-01-05,412


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('Ad Click Volume Forecasting Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rolling_w = min(SEASON_LENGTH * 2, N_POINTS // 4)
df['_rolling'] = df[TARGET].rolling(rolling_w).mean()
axes[1, 1].plot(df['date'], df['_rolling'])
axes[1, 1].set_title(f'Rolling {rolling_w}-period Mean')
df.drop(columns='_rolling', inplace=True)
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA plots saved.")

EDA plots saved.


## Target Analysis

In [7]:
print("clicks Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

clicks Statistics:
count     730.000000
mean      717.013699
std       160.216794
min       328.000000
25%       590.250000
50%       712.000000
75%       826.500000
max      1184.000000
Name: clicks, dtype: float64

CV: 0.223
Skewness: 0.269


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all points except last 28
- **Validation**: next 14
- **Test**: last 14

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree-based models handle raw features well. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek if 'D' in ['D','h','H'] else 0
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day if 'D' in ['D','h','H'] else 1
    if 'D' in ['h', 'H']:
        d['hour'] = d['date'].dt.hour
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 13 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:      130.9 | RMSE:      140.4 | MAPE: 12.99%
Seasonal Naive (7)             | MAE:       33.8 | RMSE:       49.0 | MAPE:  3.22%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
Empty DataFrame
Columns: []
Index: []


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:       78.8 | RMSE:      101.7 | MAPE:  7.24%
FLAML Test (catboost)          | MAE:       73.9 | RMSE:       86.7 | MAPE:  7.02%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]

sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))

print("StatsForecast complete.")

SF AutoARIMA                   | MAE:       31.2 | RMSE:       41.7 | MAPE:  2.99%
SF AutoETS                     | MAE:       28.1 | RMSE:       39.7 | MAPE:  2.70%
SF SeasonalNaive               | MAE:       36.2 | RMSE:       52.2 | MAPE:  3.45%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model        MAE       RMSE      MAPE
           SF AutoETS  28.090834  39.741130  2.696499
         SF AutoARIMA  31.180468  41.664898  2.990993
   Seasonal Naive (7)  33.785714  48.967190  3.222540
     SF SeasonalNaive  36.214287  52.150195  3.454802
FLAML Test (catboost)  73.921638  86.681446  7.019595
     FLAML (catboost)  78.775407 101.713566  7.239504
   Naive (Last Value) 130.928571 140.433767 12.988636

Top 3:
             model       MAE      RMSE     MAPE
        SF AutoETS 28.090834 39.741130 2.696499
      SF AutoARIMA 31.180468 41.664898 2.990993
Seasonal Naive (7) 33.785714 48.967190 3.222540


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: 73.92, Std: 45.27


## Interpretation and Business Insight

- Click volume correlates with impression volume but has different patterns
- Q4 shows significantly higher clicks due to holiday shopping intent
- Weekday clicks are consistently higher than weekend clicks
- Growth trend reflects both user growth and improving ad relevance
- CTR (clicks/impressions) is more stable than raw click volume

## Limitations

1. Synthetic data — real clicks include fraud, bots, accidental clicks
2. No CTR modeling — clicks without impressions context is incomplete
3. No campaign-level granularity
4. Click quality (conversions) not addressed

## How to Improve This Project

1. Model CTR (click-through rate) instead of raw clicks
2. Add impression forecasts as input features
3. Implement click fraud anomaly detection
4. Segment by ad format and position
5. Add conversion prediction downstream

## Production Considerations

- Real-time click monitoring with hourly updates
- Fraud detection baseline from forecast deviations
- Revenue forecasting integration (clicks × avg CPC)
- Advertiser-facing dashboards with predicted performance

## Common Mistakes

1. Forecasting clicks without considering impression supply
2. Not distinguishing valid clicks from fraudulent ones
3. Ignoring seasonal CTR changes when building features
4. Using overall click volume without campaign-level segmentation

## Mini Challenge / Exercises

1. Compute a synthetic CTR series and forecast that instead
2. Add an impression feature and see if it improves click forecasts
3. Simulate a click fraud spike and test anomaly detection
4. Try hierarchical forecasting: clicks by ad format

## Final Summary / Key Takeaways

1. Ad click volume has weekly and seasonal (Q4) patterns
2. Click forecasting is foundational for CPC revenue projections
3. Lag features capture the weekly cycle effectively
4. Q4 shopping season creates predictable click increases
5. Real-world click forecasting must account for fraud and quality

In [18]:
import json
metrics = {
    'project': 'Ad Click Volume Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Ad Click Volume Forecasting — Complete ===")

Metrics saved.

=== Ad Click Volume Forecasting — Complete ===
